# Kronos-NSE — evaluation grid on Kaggle

> Research/education tool — scenario visualization, not investment advice.

Runs the A3 evaluation grid on a Kaggle T4/P100 (16 GB), which finishes in roughly 1–2 hours
against 4–6 on a 6 GB laptop — and without the GPU dropping off the bus.

## Before you run anything

In the notebook sidebar:

1. **Accelerator → GPU T4 x2** (or P100). Without this the grid will not finish.
2. **Internet → On.** Needed to clone the repo and download the model weights.
3. **Attach the corpus dataset** — see the next cell.

## Getting `data/` here

`data/` is gitignored, so it does not arrive with the clone. On your local machine, zip
`data/` (17.6 MB) and upload it as a **Kaggle Dataset** (Datasets → New Dataset), then
attach it to this notebook. It will appear under `/kaggle/input/<your-dataset-name>/`.

**If you corrected the corpus locally, re-upload it.** The notebook fingerprints the
attached dataset against `phase-a/eval/golden.json` before doing anything expensive, so a
stale dataset stops the run in seconds rather than failing the port check hours later. Zip
`data/` again and push it as a **new version** of the Kaggle dataset.

**Do not re-run `fetch_nse.py` here.** Canonical prices are split/bonus back-adjusted, and
back-adjustment rewrites history — a corpus fetched on a different date is a *different
corpus*. Nothing would error; the numbers would simply stop being comparable to every
measurement taken so far.

## How to run it

Use **Save Version -> Save & Run All (Commit)**, not the interactive editor. An
interactive session is capped at a shorter idle timeout and dies when the browser tab
does; a committed run executes headlessly to the 12-hour limit and keeps its output.
Set `LIMIT = 60` for a first pass, confirm the port check passes, then commit the full
grid.

Every cell raises on failure, so a committed run that reports success really did
finish the grid.

## A note on exact reproduction

The baseline numbers in the verification cell are pure NumPy and **must** match exactly —
they prove the corpus and harness survived the move. The Kronos ensembles may differ in
the last digits from a different GPU architecture, which is normal. Do not mix devices
*within* one grid; finish a run on the hardware it started on, or resume on the same kind.

In [ ]:
import glob
import json
import os
import pathlib
import shutil
import subprocess
import sys

# ---- configure -------------------------------------------------------------
REPO_URL = "https://github.com/neopentane7/kronos-candlecast.git"
DATA_DIR = None  # e.g. "/kaggle/input/kronos-nse-corpus"; None = autodetect

# Private repo? Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN
# (Add-ons -> Secrets). Leave this alone if the repo is public.
USE_TOKEN = True

BATCH_SIZE = 24  # 16 GB card; drop to 12 if you hit OOM
CHECKPOINT_EVERY = 5  # flush partials every N batches
SPLIT = "test"
LIMIT = None  # e.g. 60 for a quick smoke run; None = full 708-window grid

WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "kronos-candlecast"
print("working dir:", WORK)

In [ ]:
# ---- 1. clone the project and the pinned upstream --------------------------
UPSTREAM_SHA = "67b630e67f6a18c9e9be918d9b4337c960db1e9a"
UPSTREAM_URL = "https://github.com/shiyu-coder/Kronos.git"


# The token is passed through GIT_ASKPASS, never through the URL.
#
# Embedding it as https://<token>@github.com/... puts the secret in argv, and argv
# is printed by CalledProcessError -- so any clone failure dumps the token into the
# notebook output and into whatever gets pasted somewhere for help. It also lands in
# .git/config verbatim. An askpass helper leaks through neither: git reads the
# credential from a file descriptor, the URL stays clean, and nothing that prints a
# command line has the secret to print.
git_env = dict(os.environ)
git_env["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of hanging on a prompt

if USE_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        askpass = WORK / ".git-askpass"
        askpass.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) echo "$GIT_PAT" ;;\n'
            "esac\n"
        )
        askpass.chmod(0o700)
        git_env["GIT_PAT"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
        git_env["GIT_ASKPASS"] = str(askpass)
        print("using GITHUB_TOKEN from Kaggle Secrets")
    except Exception as exc:  # noqa: BLE001 - fall back to an anonymous clone
        print(f"no usable token ({type(exc).__name__}); trying anonymous clone")


def git_run(*args):
    """Run git without letting a failure echo argv or the environment."""
    p = subprocess.run(["git", *args], capture_output=True, text=True, env=git_env)
    if p.returncode:
        detail = (p.stderr or p.stdout).strip().splitlines()
        hint = ""
        if any("403" in ln or "not granted" in ln for ln in detail):
            hint = (
                "\n\nA 403 on a *clone* usually means the fine-grained token has no "
                "Contents permission, or does not list this repository under "
                "'Only select repositories'. GitHub reports both as a write-access "
                "error, which is misleading. Check: Contents -> Read-only."
            )
        elif any("could not read Username" in ln for ln in detail):
            hint = (
                "\n\nNo credential was supplied. Either the GITHUB_TOKEN secret is not "
                "attached to this notebook (Add-ons -> Secrets, toggle it on), or the "
                "repo is public and USE_TOKEN should be False."
            )
        raise SystemExit(
            f"git {args[0]} failed (exit {p.returncode}):\n  " + "\n  ".join(detail[-4:]) + hint
        )
    return p.stdout.strip()


if not REPO.exists():
    git_run("clone", "--quiet", REPO_URL, str(REPO))
print("repo:", git_run("-C", str(REPO), "log", "--oneline", "-1"))

# Upstream is gitignored by design and reproduced at its pinned commit. The harness
# overlays its generation loop and an equivalence test asserts bit-identical output,
# so the SHA is not optional.
up = REPO / "phase-a" / "Kronos"
if not up.exists():
    git_run("clone", "--quiet", UPSTREAM_URL, str(up))
    git_run("-C", str(up), "checkout", "--quiet", "--detach", UPSTREAM_SHA)
print("upstream:", git_run("-C", str(up), "rev-parse", "HEAD")[:12])

# The helper has done its job; the token should not outlive the clone.
if USE_TOKEN and (WORK / ".git-askpass").exists():
    (WORK / ".git-askpass").unlink()
    git_env.pop("GIT_PAT", None)
    git_env.pop("GIT_ASKPASS", None)

In [ ]:
# ---- 2. dependencies -------------------------------------------------------
# Kaggle already ships torch with a working CUDA build, so we install only what is
# missing rather than re-resolving the lockfile (which would pull a ~2.5 GB torch
# wheel). The trade-off: exact package pinning is relaxed.
#
# Minimum versions are pinned rather than bare names. `pip install pandera` is a
# no-op when ANY pandera is already present -- pip reports "already satisfied" and
# does not upgrade -- and Kaggle's base image ships many of these. Without the
# bounds, an old pandera would survive and `import pandera.pandas` would fail
# halfway through the run.
REQUIREMENTS = [
    "pandera>=0.24",  # the pandera.pandas namespace
    "scoringrules>=0.9",  # crps_ensemble(estimator="fair")
    "numpy>=1.22",  # np.quantile(method="weibull")
    "duckdb",
    "exchange_calendars",
    "einops",
]
# subprocess, not a `!` escape. IPython only interpolates simple names into `!`
# lines; an expression with nested quotes and a generator is passed through to bash
# verbatim, which then fails on the parenthesis and installs nothing -- while the
# cell carries on to the imports. The pipe to `tail` made it worse by discarding
# pip's exit code, so even a genuine install failure would have looked fine.
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *REQUIREMENTS],
    capture_output=True,
    text=True,
)
_out = (_pip.stdout + _pip.stderr).strip()
if _out:
    print("\n".join(_out.splitlines()[-3:]))
if _pip.returncode:
    raise SystemExit(f"pip install failed (exit {_pip.returncode}); see the output above")
print(f"installed/verified: {', '.join(REQUIREMENTS)}")

# Verify the APIs we actually call, not the version strings. A satisfied version
# constraint is not proof the function exists with the signature we use.
import numpy as np  # noqa: E402
import pandera.pandas as pa  # noqa: E402  - namespace only exists in pandera >= 0.24
import scoringrules as sr  # noqa: E402
import torch  # noqa: E402

checks = {
    "np.quantile(method='weibull')": lambda: np.quantile([1.0, 2, 3], 0.5, method="weibull"),
    "sr.crps_ensemble(estimator='fair')": lambda: sr.crps_ensemble(
        np.zeros(2), np.zeros((2, 8)), m_axis=-1, estimator="fair"
    ),
    "pa.DataFrameSchema": lambda: pa.DataFrameSchema(columns={}),
}
broken = []
for name, probe in checks.items():
    try:
        probe()
        print(f"  ok   {name}")
    except Exception as exc:  # noqa: BLE001 - report every failure, not just the first
        print(f"  FAIL {name}: {type(exc).__name__}: {exc}")
        broken.append(name)
if broken:
    raise SystemExit(f"unusable environment: {broken}. Restart the kernel and re-run.")

print(f"\ntorch {torch.__version__} | cuda {torch.version.cuda}")
print(f"cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Accelerator -> GPU T4 x2 in the sidebar.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB")

In [ ]:
# ---- 3. restore the corpus -------------------------------------------------
src = DATA_DIR
if src is None:
    hits = glob.glob("/kaggle/input/*/parquet") + glob.glob("/kaggle/input/*/data/parquet")
    if not hits:
        raise SystemExit(
            "No corpus found under /kaggle/input. Upload data/ as a Kaggle Dataset "
            "and attach it, or set DATA_DIR explicitly."
        )
    src = str(pathlib.Path(hits[0]).parent)
    print("autodetected corpus at", src)

dest = REPO / "data"
dest.mkdir(exist_ok=True)
# `if target.exists(): continue` was wrong: a session that died mid-copy leaves a
# partial data/parquet, and skipping it would hand the grid a silently truncated
# corpus. copytree(dirs_exist_ok=True) fills in what is missing instead.
for item in pathlib.Path(src).iterdir():
    target = dest / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# 59 tickers, one partition each. An exact count, not `> 0`: a truncated corpus is
# the failure mode that produces plausible-looking numbers that mean nothing, and
# the port check in the next cell only catches it because the baselines shift.
EXPECTED_PARTITIONS = 59
n_parts = len(list((dest / "parquet").glob("*/*.parquet")))
print(f"corpus restored: {n_parts} ticker partitions")
assert n_parts == EXPECTED_PARTITIONS, (
    f"expected {EXPECTED_PARTITIONS} parquet partitions, found {n_parts} -- the "
    "dataset is truncated or its layout differs from the local corpus"
)

# The numbers this notebook asserts are a property of the corpus, and the corpus is
# gitignored -- it arrives as a dataset you uploaded, on its own schedule. A dataset
# predating the corpus correction would reproduce the SUPERSEDED numbers perfectly and
# fail the port check looking like a harness bug, hours into a session. Check identity
# first, before the model weights are even downloaded.
sys.path.insert(0, str(REPO))
from common.corpus import check as check_corpus  # noqa: E402

GOLDEN = json.loads((REPO / "phase-a" / "eval" / "golden.json").read_text())
check_corpus(dest / "parquet", GOLDEN["corpus"])
print(f"corpus fingerprint OK: {GOLDEN['corpus']['digest']} ({GOLDEN['corpus']['n_rows']} rows)")

In [ ]:
# ---- 4. VERIFY THE PORT ----------------------------------------------------
# Baselines are pure NumPy and deterministic from the grid and the seed, so these
# numbers must reproduce exactly on any machine. If they do, the corpus, window
# enumeration, metric layer and seeding all survived the move. If they differ, the
# corpus is not the same one -- almost certainly because fetch_nse.py was re-run.
#
# This asserts rather than printing an expected block for a human to eyeball: a
# check you have to read is a check that gets skipped when you are in a hurry.
os.chdir(REPO)
before = set(glob.glob("results/*/results.json"))
r = subprocess.run(
    [
        sys.executable,
        "phase-a/eval/calibrate.py",
        "--split",
        "test",
        "--skip-model",
        "--no-figures",
    ],
    capture_output=True,
    text=True,
)
print(r.stdout[-1200:] or r.stderr[-1200:])
if r.returncode != 0:
    raise SystemExit(f"the baseline run failed (exit {r.returncode}); see the output above")

# Read from the committed golden file rather than restating the numbers here. Held in
# two places they drift, and the drift surfaces as a failed cloud session instead of a
# failed test. tests/test_golden.py asserts the same file against the same harness.
EXPECTED = GOLDEN["models"]
EXPECTED_BLOCKS = GOLDEN["effective_blocks"]
TOL = GOLDEN["tolerances"]
print(f"golden: run {GOLDEN['generated_from_run']} @ {GOLDEN['git_sha']}")

# Bind to the run this cell just produced, not to the newest on disk. Sorting by
# name picks whatever ran last, so a restored results/ or a re-run of this cell
# could pass the check against an *older* run while the new one silently failed.
new = set(glob.glob("results/*/results.json")) - before
if len(new) != 1:
    raise SystemExit(f"expected exactly one new run directory, got {sorted(new)}")
run_json = pathlib.Path(next(iter(new)))
print("checking", run_json.parent)
res = json.loads(run_json.read_text())

failures = []
for name, want in EXPECTED.items():
    got = res["models"][name]
    if abs(got["crps"] - want["crps"]) > TOL["crps"]:
        failures.append(f"{name} CRPS {got['crps']:.4f} != {want['crps']}")
    cov = got["coverage"]["80"]["empirical"]
    if abs(cov - want["coverage_80"]) > TOL["coverage"]:
        failures.append(f"{name} cov@80 {cov:.4f} != {want['coverage_80']}")

blocks = res["models"]["last_value"]["effective_blocks"]
if blocks != EXPECTED_BLOCKS:
    failures.append(f"effective blocks {blocks} != {EXPECTED_BLOCKS}")

if failures:
    raise SystemExit("PORT CHECK FAILED:\n  " + "\n  ".join(failures))
print("\nPORT CHECK PASSED — corpus, windows, metrics and seeding all reproduce.")

In [ ]:
# ---- 4b. OPTIONAL: prove determinism and resume ON THIS GPU (~20 min) ------
# tests/test_resume.py asserts resume is bit-identical, but it does that against a
# fake sampler on the CPU: it proves the bookkeeping, not that this card reproduces
# its own sampling. Kronos draws tokens with torch.multinomial on the GPU, and
# whether two identically-seeded runs agree bit-for-bit is a property of the
# hardware and the kernel selection, not of anything in this repo. Fifteen minutes
# here decides whether a grid that spans two sessions is one measurement or two.
#
# Three runs, because two different things can break and they need separating:
#   A  -- 60 windows, clean
#   A2 -- the same command again, fresh process. A != A2 means this GPU does not
#         reproduce its own sampling, and no resume logic could rescue that.
#   B  -- the same 60, killed at the first checkpoint and resumed. A != B while
#         A == A2 means the resume bookkeeping is at fault, not the hardware.
# Diagnosing those separately is worth the extra five minutes; a single A-vs-B
# comparison cannot tell them apart.
#
# All three arms run at the PRODUCTION BATCH_SIZE, deliberately. cuBLAS and cuDNN pick
# kernels by tensor shape through runtime heuristics, and the split-K variants that
# accumulate with atomics -- the classic source of run-to-run nondeterminism -- appear
# only at particular shapes. Bit-identity at batch 6 would therefore certify nothing
# about batch 24. A cheap smoke config could pass while the grid's actual kernel
# population is nondeterministic.
import time  # noqa: E402

RUN_TWIN_CHECK = True
TWIN_LIMIT = 60
TWIN_CHECKPOINT_EVERY = 1
KILL_TIMEOUT_SECONDS = 900  # give up waiting for a checkpoint after this long

# The arms write outside results/ so the downloaded archive contains exactly one run --
# the grid. Otherwise three throwaway directories ride along in the zip and sit inside
# the globs cells 6 and 7 resolve with sorted(...)[-1]. Timestamp ordering makes the grid
# win today, which is precisely the kind of correctness that survives until someone
# reorders a glob.
TWIN_ROOT = pathlib.Path("/kaggle/working/twin_runs")

if not RUN_TWIN_CHECK:
    print("twin check skipped")
else:
    # Arm B is only a resume test if the run flushes at least once AND still has work
    # left when it is killed. At TWIN_LIMIT=60 and BATCH_SIZE=24 there are 3 batches,
    # so a production cadence of 5 would never flush and B would resume nothing while
    # still reporting a pass. Checked rather than assumed.
    _n_batches = -(-TWIN_LIMIT // BATCH_SIZE)
    assert _n_batches >= 2, (
        f"TWIN_LIMIT={TWIN_LIMIT} at BATCH_SIZE={BATCH_SIZE} is {_n_batches} batch(es); "
        "arm B needs at least 2 so the kill leaves work to resume"
    )
    assert _n_batches > TWIN_CHECKPOINT_EVERY or TWIN_CHECKPOINT_EVERY == 1, (
        f"TWIN_CHECKPOINT_EVERY={TWIN_CHECKPOINT_EVERY} never fires in {_n_batches} "
        "batches, so nothing would be flushed to resume from"
    )
    print(
        f"twin arms: {_n_batches} batches of {BATCH_SIZE}, flush every "
        f"{TWIN_CHECKPOINT_EVERY} -- same shapes the full grid will use"
    )
    twin_cmd = [
        sys.executable,
        "phase-a/eval/calibrate.py",
        "--split",
        SPLIT,
        "--limit",
        str(TWIN_LIMIT),
        "--batch-size",
        str(BATCH_SIZE),
        "--checkpoint-every",
        str(TWIN_CHECKPOINT_EVERY),
        "--no-figures",
    ]

    TWIN_ROOT.mkdir(parents=True, exist_ok=True)
    twin_env = {**os.environ, "KRONOS_RESULTS_ROOT": str(TWIN_ROOT)}

    def _dirs():
        return {p for p in TWIN_ROOT.glob("*") if p.is_dir()}

    def _clean_run(label):
        seen = _dirs()
        p = subprocess.run(twin_cmd, capture_output=True, text=True, env=twin_env)
        if p.returncode != 0:
            raise SystemExit(f"twin run {label} failed:\n" + (p.stdout + p.stderr)[-2000:])
        d = pathlib.Path(next(iter(_dirs() - seen)))
        print(f"run {label}:", d)
        return d

    dir_a = _clean_run("A")
    dir_a2 = _clean_run("A2")

    # Kill on the first flushed checkpoint rather than after a fixed wait. A fixed
    # wait lands either before the first flush or after the last one depending on how
    # fast the card is, and in both cases the "resume" resumes nothing.
    seen = _dirs()
    log = pathlib.Path("/kaggle/working/twin_run_b.log")
    dir_b = None
    with log.open("w") as fh:
        proc = subprocess.Popen(
            twin_cmd, stdout=fh, stderr=subprocess.STDOUT, text=True, env=twin_env
        )
        deadline = time.time() + KILL_TIMEOUT_SECONDS
        while time.time() < deadline and proc.poll() is None:
            if dir_b is None:
                fresh = _dirs() - seen
                dir_b = pathlib.Path(next(iter(fresh))) if fresh else None
            if dir_b is not None and any(dir_b.glob("partial_*.npy")):
                break
            time.sleep(5)
        proc.terminate()  # stands in for the session loss this whole design exists for
        proc.wait()

    if dir_b is None:
        fresh = _dirs() - seen
        if not fresh:
            raise SystemExit(f"run B produced no run directory; see {log}")
        dir_b = pathlib.Path(next(iter(fresh)))
    partials = sorted(p.name for p in dir_b.glob("partial_*.npy"))
    print(f"run B interrupted; partials on disk: {partials}")
    if not partials:
        raise SystemExit(
            "no checkpoint was flushed within "
            f"{KILL_TIMEOUT_SECONDS}s, so a resume would resume nothing and this would "
            f"prove nothing. Check {log} before raising the timeout."
        )

    b = subprocess.run(
        twin_cmd + ["--resume", str(dir_b)], capture_output=True, text=True, env=twin_env
    )
    if b.returncode != 0:
        raise SystemExit("resume failed:\n" + (b.stdout + b.stderr)[-2000:])
    print("run B resumed to completion:", dir_b)

    def _diff(left, right, label):
        el, er = np.load(left / "ensembles.npz"), np.load(right / "ensembles.npz")
        bad = []
        print(f"\n  {label}")
        for k in [k for k in el.files if k.startswith("ens__")]:
            same = np.array_equal(el[k], er[k])
            delta = float(np.abs(el[k].astype("float64") - er[k].astype("float64")).max())
            print(f"    {k:<28} identical={same}  max diff={delta:.3e}")
            if not same:
                bad.append(k)
        return bad

    nondeterministic = _diff(dir_a, dir_a2, "A vs A2  (does this GPU reproduce itself?)")
    mismatched = _diff(dir_a, dir_b, "A vs B   (is resume exact?)")

    peak = json.loads((dir_a / "results.json").read_text()).get("peak_vram_mb")
    if peak:
        print(
            f"\npeak VRAM at batch {BATCH_SIZE}: {peak:.0f} MiB of "
            f"{props.total_memory / 2**20:.0f} MiB"
        )

    if nondeterministic:
        raise SystemExit(
            "THIS GPU DOES NOT REPRODUCE ITS OWN SAMPLING: "
            + ", ".join(nondeterministic)
            + ".\nTwo identical commands, same seed, different draws -- so kernel "
            "selection or reduction order is varying between processes. Resume cannot be "
            "bit-identical when a clean re-run is not, and this is not a harness bug. "
            "Finish the grid in one session and record the finding; it is a real property "
            "of this hardware and worth reporting."
        )
    if mismatched:
        raise SystemExit(
            "RESUME IS NOT EXACT: "
            + ", ".join(mismatched)
            + ".\nA clean re-run reproduced (A == A2), so the hardware is fine and the "
            "resume bookkeeping is at fault. Do not run a grid across sessions until this "
            "is understood -- the halves would be two different forecasters in one table."
        )
    print(
        "\nTWIN CHECK PASSED - this GPU reproduces its own sampling (A == A2) and "
        "resume is exact (A == B)."
    )

In [ ]:
# ---- 5. run the grid -------------------------------------------------------
# Resumable: if a previous session left a run directory with a partial, point --resume
# at it and only the missing batches are recomputed. Resume is bit-identical, because
# each batch is seeded from its own offset.
RESUME_DIR = None  # e.g. "results/20260804T...."; None = fresh run

cmd = [
    sys.executable,
    "phase-a/eval/calibrate.py",
    "--split",
    SPLIT,
    "--batch-size",
    str(BATCH_SIZE),
    "--checkpoint-every",
    str(CHECKPOINT_EVERY),
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if RESUME_DIR:
    cmd += ["--resume", RESUME_DIR]

print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
print("exit code:", code)
if code:
    # Without this the analysis and packaging cells run on whatever happens to be in
    # results/, which under Save & Run All means a green notebook over a failed grid.
    raise SystemExit(code)

In [ ]:
# ---- 6. offline analysis (CPU only) ----------------------------------------
# subprocess rather than a `!` escape: shell escapes inside an if-block are fragile
# and do not survive tooling that parses the notebook as Python.
runs = sorted(glob.glob("results/*/ensembles.npz"))
if not runs:
    print("no ensembles.npz yet — run the grid cell first")
else:
    latest = str(pathlib.Path(runs[-1]).parent)
    print("analysing", latest, flush=True)
    r = subprocess.run(
        [sys.executable, "phase-a/eval/run_analysis.py", latest],
        capture_output=True,
        text=True,
    )
    print(r.stdout or r.stderr)

In [ ]:
# ---- 7. package results for download ---------------------------------------
# Only /kaggle/working survives the session. Zip results/ so it can be downloaded
# from the notebook's Output tab, then copied back into the local repo.
out = "/kaggle/working/kronos_results"
shutil.make_archive(out, "zip", root_dir=str(REPO), base_dir="results")
size = os.path.getsize(out + ".zip") / 2**20
print(f"{out}.zip  ({size:.1f} MB)")
print("\nDownload from the Output panel, unzip into the local repo, then:")
print("  uv run python phase-a/eval/run_analysis.py results/<run-dir>")

## If the session ends mid-run

Nothing is lost. Partials flush every `CHECKPOINT_EVERY` batches.

1. Download `kronos_results.zip` from the Output panel before the session expires.
2. In the next session, re-run steps 1–4, upload the zip as a dataset (or re-clone and
   restore `results/`), set `RESUME_DIR` to the run directory, and re-run step 5.
   Leave `BATCH_SIZE` alone: batches are seeded from their own offset, so changing it
   would reseed every remaining window. The harness refuses such a resume rather than
   splicing two forecasters into one grid.
3. Step 4b is worth running once per GPU type, and never again after it passes.

Only the missing batches are recomputed, and the result is identical to an uninterrupted
run — `tests/test_resume.py` asserts that at `rtol=0, atol=0`.

## Tuning

| symptom | change |
|---|---|
| CUDA out of memory | `BATCH_SIZE` 24 → 12 → 6 |
| Want a quick smoke test first | `LIMIT = 60` (~5 min) |
| Session keeps dying | lower `CHECKPOINT_EVERY` to 2 |

Peak VRAM was 5.9 GB at batch 6 on a 6 GB card, so batch 24 on a 16 GB card should sit
comfortably inside budget.